In [1]:
"""
need to pip install a few modules:
    pandas
    scikit-learn
    matplotlib

"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.multiclass import OneVsRestClassifier
from xgboost import XGBClassifier

In [2]:
#load data
data_df = pd.read_csv("data/train.csv")
test_df = pd.read_csv("data/test.csv")

In [7]:
#view data
total_len = len(data_df)  # three possible classifications: GALAXY, STAR, QSO
# data_df.head(10)
# test_df.head(10)

In [ ]:
#convert categorical variables to numeric dummy variables and do a test-train split
import model_helper_functions
# import importlib
# importlib.reload(model_helper_functions)
if(True):
    cleaned_df = model_helper_functions.cat_to_dummy(data_df,False)
    X_train, X_test, y_train, y_test = train_test_split(cleaned_df.drop(columns=['class']), cleaned_df['class'], test_size=0.25, random_state = 12345)
else: #no need to convert categorical to dummy encoding
    X_train, X_test, y_train, y_test = train_test_split(data_df.drop(columns=['class']), data_df['class'], test_size=0.25, random_state = 12345)

In [ ]:
print("Length of training data split: ", len(X_train), "and", len(y_train))
print("Length of testing section: ", len(X_test), "and", len(y_test))

Length of training data split:  433010 and 433010
Length of testing section:  144337 and 144337


104969    0
482690    1
63041     1
460495    0
324763    0
Name: class, dtype: int64

In [5]:
###
# XGboost code, copied from https://xgboost.readthedocs.io/en/stable/get_started.html
# ToDo: hyperparameter tuning, read up on what is actually happening behind the scenes
###

bst = XGBClassifier(n_estimators=2, max_depth=2, learning_rate=1)
# fit model
bst.fit(X_train, y_train)

##
# ToDo: visualize more
##

,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [ ]:
model_helper_functions.save_model_results(bst, X_test, y_test,"xgboost_validation")
model_helper_functions.save_model_results(bst, X_train, y_train,"xgboost_fit")

In [ ]:
###
# Logistic regression code
###
# Wrap your model to explicitly enforce One-Vs-Rest
first_model = OneVsRestClassifier(LogisticRegression(solver='liblinear'))
first_model.fit(X_train, y_train)

,"estimator estimator: estimator objectA regressor or a classifier that implements :term:`fit`.When a classifier is passed, :term:`decision_function` will be usedin priority and it will fallback to :term:`predict_proba` if it is notavailable.When a regressor is passed, :term:`predict` is used.",LogisticRegre...r='liblinear')
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation: the `n_classes`one-vs-rest problems are computed in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: 0.20 `n_jobs` default changed from 1 to None",None
,"verbose verbose: int, default=0The verbosity level, if non zero, progress messages are printed.Below 50, the output is sent to stderr. Otherwise, the output is sentto stdout. The frequency of the messages increases with the verbositylevel, reporting all iterations at 10. See :class:`joblib.Parallel` formore details... versionadded:: 1.1",0
Name,Type,Value
"classes_ classes_: array, shape = [`n_classes`]Class labels.","ndarray[int64](3,)","[0,1,2]"
estimators_ estimators_: list of `n_classes` estimatorsEstimators used for predictions.,list,"[LogisticRegre...r='liblinear'), LogisticRegre...r='liblinear'), LogisticRegre...r='liblinear')]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](14,)","['alpha','delta','u',...,'O/B','Blue_Cloud','Red_Sequence']"
label_binarizer_ label_binarizer_: LabelBinarizer objectObject used to transform multiclass labels to binary labels andvice-versa.,LabelBinarizer,LabelBinarize...e_output=True)
multilabel_ multilabel_: booleanWhether a OneVsRestClassifier is a multilabel classifier.,bool,False
n_classes_ n_classes_: intNumber of classes.,int,3
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 0.24,int,14


In [56]:
# Extract weights from all classes in the model
weights = np.array([estimator.coef_[0] for estimator in first_model.estimators_])

# Extract biases (intercepts) for all classes
biases = np.array([estimator.intercept_[0] for estimator in first_model.estimators_])

for target_class, w, b in zip(first_model.classes_, weights, biases):
    print(f"Class {target_class} -> Bias: {b:.4f} | Weights: {w}")

Class 0 -> Bias: 0.5822 | Weights: [-1.20065575e-03 -7.26509775e-03 -1.02059936e-01  1.85289991e+00
  1.12831799e+00 -1.36197072e+00 -1.66661848e+00 -5.02869988e-01
 -2.41841000e-02  4.24709811e-01  4.49768186e-01 -2.68076901e-01
  8.17602259e-02  5.00456770e-01]
Class 1 -> Bias: -1.3644 | Weights: [ 1.98115232e-03  2.33751471e-02 -1.77940087e+00 -5.45736138e-01
 -7.19201450e-01  1.92259052e+00  1.22870767e+00  3.39998569e+00
 -4.22819704e-02 -5.30278996e-01 -8.96437800e-01  1.04630335e-01
 -1.57367562e+00  2.09307190e-01]
Class 2 -> Bias: -1.6506 | Weights: [-3.49389389e-04 -7.20868066e-03  4.28845548e-01 -1.57427274e+00
  1.81003303e-01  2.81879335e-01  9.46722738e-01 -1.23186365e+01
 -2.24640908e-01 -2.11942988e-01 -4.07774241e-01 -8.06276398e-01
 -2.45144590e-01 -1.40548995e+00]
